# Enterprise Email Triage Agent - RL Training

Training a Llama-3-8B model using Unsloth and TRL for the Enterprise Email Triage environment.

## Overview
- **Model**: Llama-3-8B-Instruct with 4-bit quantization
- **Environment**: EnterpriseEmailEnv with LLM tool calling
- **Framework**: TRL (Transformers Reinforcement Learning)
- **Optimization**: LoRA adapters with r=16
- **Target**: Learn optimal email triage decisions

In [ ]:
# Cell 1: Dependencies for Google Colab
print("Installing dependencies for Google Colab...")

# Install Unsloth for efficient training
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install core ML libraries
!pip install trl peft torch transformers accelerate bitsandbytes

# Install environment dependencies
!pip install openenv-core pydantic

# Install additional utilities
!pip install datasets tensorboard tqdm

print("✅ All dependencies installed successfully!")

In [ ]:
# Cell 2: Model & Tokenizer Setup with Unsloth
import torch
from unsloth import FastLanguageModel
import json
import re
import numpy as np
from typing import Dict, Any, List
import warnings
warnings.filterwarnings("ignore")

print("🚀 Setting up model with Unsloth optimization...")

# Load model with 4-bit quantization (optimized for Colab)
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/llama-3-8b-Instruct-bnb-4bit",
    load_in_4bit=True,
    device_map="auto",
    max_seq_length=2048,
)

# Configure LoRA adapters for efficient training
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", 
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"📊 Model parameters: {model.num_parameters():,}")
print("✅ Model setup complete!")

In [ ]:
# Cell 3: Environment & Reward System Setup
import sys
import os
from typing import Dict, Any, List

# Download environment files from GitHub (for Colab)
!wget -q https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/env.py
!wget -q https://raw.githubusercontent.com/vaishali-strategy/email-triage/master/reward_system.py

# Import environment and reward system
from env import EnterpriseEmailEnv
from reward_system import reward_system

print("🔧 Initializing environment...")

# Initialize environment
env = EnterpriseEmailEnv()
observation = env.reset()

print(f"✅ Environment initialized with {len(env.emails)} emails")
print(f"📧 Current email: {observation['current_email']['subject']}")

def format_observation_to_prompt(observation: Dict[str, Any]) -> str:
    """
    Convert environment observation to LLM prompt format.
    """
    current_email = observation['current_email']
    
    system_prompt = """You are an Enterprise Email Triage Agent. Analyze emails and decide the best action.

Available Actions:
1. auto_reply - Automatically respond with a message (good for routine inquiries)
2. route_to_human - Forward to human agent (required for urgent/sensitive issues)  
3. ask_for_clarification - Request more information (good for ambiguous emails)

Departments: IT, Customer Service, Emergency Support, HR, Security, Finance

Rules:
- VIP emergencies → Route to Emergency Support
- Phishing attempts → Route to Security
- HR sensitive matters → Route to HR
- Password resets → Auto-reply with helpful message
- When unsure → Ask for clarification

Respond with JSON format:
{"tool": "action_name", "arguments": {"email_id": "email_id", "parameter": "value"}}"""
    
    user_prompt = f"""Email Details:
From: {current_email['sender']}
Subject: {current_email['subject']}
Body: {current_email['body']}
Priority: {current_email['priority']}/5
VIP Status: {current_email['is_vip']}
Intent: {current_email['intent']}

Decide the best action and respond with JSON:"""
    
    # Format for Llama-3 chat template
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

In [ ]:
# Cell 4: Enhanced Rollout Collection with Reward System
from tqdm import tqdm
import time
from datasets import Dataset

def collect_rollouts(env, model, tokenizer, num_episodes=5, max_steps_per_episode=10):
    """
    Collect rollout data with enhanced reward system.
    """
    rollout_data = []
    episode_rewards = []
    
    print(f"🔄 Collecting {num_episodes} episodes...")
    
    for episode in tqdm(range(num_episodes), desc="Episodes"):
        obs = env.reset()
        episode_data = []
        episode_reward = 0
        
        for step in range(max_steps_per_episode):
            # Format observation for model
            prompt = format_observation_to_prompt(obs)
            
            # Generate action from model
            inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            
            # Extract and parse JSON action
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Extract JSON from generated text
            json_match = re.search(r'\{.*\}', generated_text, re.DOTALL)
            if json_match:
                try:
                    action = json.loads(json_match.group())
                    # Validate action format
                    if "tool" in action and "arguments" in action:
                        if "email_id" not in action["arguments"]:
                            action["arguments"]["email_id"] = obs['current_email']['email_id']
                    else:
                        action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
                except:
                    action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
            else:
                action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
            
            # Take action in environment
            next_obs, reward, done, info = env.step(action)
            
            # Get detailed reward breakdown
            reward_breakdown = reward_system.calculate_reward(obs['current_email'], action)
            
            # Store transition
            episode_data.append({
                "prompt": prompt,
                "action": action,
                "reward": reward_breakdown.total_reward,
                "reward_breakdown": reward_breakdown,
                "observation": next_obs
            })
            
            episode_reward += reward_breakdown.total_reward
            
            # Progress tracking
            print(f"  Step {step+1}: {action['tool']} → Reward: {reward_breakdown.total_reward:+.2f} | Total: {episode_reward:+.2f}")
            
            obs = next_obs
            
            if done:
                break
        
        rollout_data.extend(episode_data)
        episode_rewards.append(episode_reward)
        
        print(f"📊 Episode {episode+1} complete: Total reward = {episode_reward:+.2f}")
    
    print(f"✅ Collected {len(rollout_data)} transitions")
    print(f"📈 Average episode reward: {np.mean(episode_rewards):+.2f}")
    
    return rollout_data, episode_rewards

# Test rollout collection
print("🧪 Testing rollout collection...")
test_rollouts, test_rewards = collect_rollouts(env, model, tokenizer, num_episodes=2, max_steps_per_episode=3)
print(f"Test successful: {len(test_rollouts)} transitions collected")

In [ ]:
# Cell 5: PPO Training Setup
from trl import PPOTrainer, PPOConfig
from transformers import TrainingArguments
import torch.nn.functional as F

print("🎯 Setting up PPO training...")

# PPO Configuration
ppo_config = PPOConfig(
    batch_size=4,
    mini_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1.4e-5,
    max_grad_norm=1.0,
    optimize_cuda_cache=True,
    ratio_threshold=10.0,
    log_with="tensorboard",
    project_kwargs={"project_name": "email-triage-rl", "tags": ["llama3", "email-triage", "rl"]},
)

# Training Arguments
training_args = TrainingArguments(
    output_dir="./email_trainer_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1.4e-5,
    logging_steps=5,
    save_steps=50,
    num_train_epochs=3,
    fp16=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="tensorboard",
    save_total_limit=2,
)

def prepare_dataset(rollout_data):
    """Prepare dataset for PPO training."""
    dataset_dict = {
        "query": [item["prompt"] for item in rollout_data],
        "response": [json.dumps(item["action"]) for item in rollout_data],
        "reward": [item["reward"] for item in rollout_data],
    }
    return Dataset.from_dict(dataset_dict)

print("✅ PPO configuration complete")

In [ ]:
# Cell 6: Training Execution
print("🚀 Starting RL training...")

# Collect initial rollouts
print("\n📊 Collecting training rollouts...")
rollout_data, episode_rewards = collect_rollouts(env, model, tokenizer, num_episodes=8, max_steps_per_episode=10)

# Prepare dataset
train_dataset = prepare_dataset(rollout_data)
print(f"📋 Dataset prepared with {len(train_dataset)} examples")

# Initialize PPO trainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=None,  # Using same model as reference for simplicity
    tokenizer=tokenizer,
    dataset=train_dataset,
    args=training_args,
)

# Training loop
print("\n🏋️ Starting PPO training...")
training_stats = ppo_trainer.train()

print("✅ Training completed!")
print(f"📈 Final average reward: {np.mean(episode_rewards):+.2f}")

# Show training summary
print("\n📊 Training Summary:")
print(f"- Total transitions: {len(rollout_data)}")
print(f"- Episodes: {len(episode_rewards)}")
print(f"- Average reward: {np.mean(episode_rewards):+.3f}")
print(f"- Reward improvement: {episode_rewards[-1] - episode_rewards[0]:+.3f}")

# Cell 7: Save Model & Evaluation
print("💾 Saving trained model...")

# Create output directory
!mkdir -p ./trained_email_triage_agent

# Save LoRA adapter (correct way for LoRA/QLoRA)
adapter_path = "./trained_email_triage_agent"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"✅ Model saved to {adapter_path}")

# Test trained model
print("\n🧪 Testing trained model...")
test_episodes = 5
test_rewards = []

for i in range(test_episodes):
    obs = env.reset()
    prompt = format_observation_to_prompt(obs)
    
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,  # Lower temperature for evaluation
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_match = re.search(r'\{.*\}', generated_text, re.DOTALL)
    
    if json_match:
        try:
            action = json.loads(json_match.group())
            if "email_id" not in action["arguments"]:
                action["arguments"]["email_id"] = obs['current_email']['email_id']
        except:
            action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
    else:
        action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
    
    # Get reward
    reward_breakdown = reward_system.calculate_reward(obs['current_email'], action)
    test_rewards.append(reward_breakdown.total_reward)
    
    print(f"Test {i+1}: {action['tool']} → Reward: {reward_breakdown.total_reward:+.2f}")

print(f"\n📊 Test Results:")
print(f"- Average test reward: {np.mean(test_rewards):+.3f}")
print(f"- Test reward std: {np.std(test_rewards):+.3f}")

# Download for local use
from google.colab import files
print("\n📦 Downloading trained model...")
!zip -r trained_email_triage_agent.zip trained_email_triage_agent/
files.download("trained_email_triage_agent.zip")

print("🎉 Training complete! Model ready for deployment!")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Extract logs from the TRL trainer
history = trainer.state.log_history
df = pd.DataFrame(history)

# 1. Plot and save Loss Curve
if 'loss' in df.columns:
    plt.figure(figsize=(8, 5))
    df.dropna(subset=['loss']).plot(x='step', y='loss', title='Training Loss')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.savefig('loss_curve.png')
    plt.close()

# 2. Plot and save Reward Curve
if 'eval_reward' in df.columns or 'reward' in df.columns: # GRPO metric name varies slightly
    reward_col = 'eval_reward' if 'eval_reward' in df.columns else 'reward'
    plt.figure(figsize=(8, 5))
    df.dropna(subset=[reward_col]).plot(x='step', y=reward_col, title='Agent Reward Over Time', color='green')
    plt.xlabel('Steps')
    plt.ylabel('Reward')
    plt.savefig('reward_curve.png')
    plt.close()

print("✅ Saved loss_curve.png and reward_curve.png. Download these and commit them to GitHub!")